# 🔥 Phase 3c: Burnout Risk Detection
Classification — High vs Low Burnout Risk

In [ ]:
import pandas as pd, numpy as np, warnings, pickle, os
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve
from xgboost import XGBClassifier
warnings.filterwarnings('ignore')

X_train = pd.read_csv('data/X_train_burnout.csv')
X_test  = pd.read_csv('data/X_test_burnout.csv')
y_train = pd.read_csv('data/y_train_burnout.csv').squeeze()
y_test  = pd.read_csv('data/y_test_burnout.csv').squeeze()
print("✅ Data loaded | Burnout ratio:", f"{y_train.mean():.2%}")


In [ ]:
models = {
    'Random Forest':     RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1),
    'XGBoost':           XGBClassifier(n_estimators=200, random_state=42, eval_metric='logloss', verbosity=0),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=200, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:,1]
    from sklearn.metrics import accuracy_score
    acc = accuracy_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_prob)
    results[name] = {'model':model, 'pred':y_pred, 'prob':y_prob, 'acc':acc, 'auc':auc}
    print(f"{name:22s} | Accuracy: {acc:.4f} | AUC: {auc:.4f}")

best_name = max(results, key=lambda k: results[k]['auc'])
print(f"\n🏆 Best Model: {best_name}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Confusion Matrix
cm = confusion_matrix(y_test, results[best_name]['pred'])
sns.heatmap(cm, annot=True, fmt='d', cmap='Reds', ax=axes[0],
            xticklabels=['Low Risk','High Risk'], yticklabels=['Low Risk','High Risk'])
axes[0].set_title(f'Confusion Matrix\n{best_name}', fontweight='bold')

# ROC
for name, res in results.items():
    fpr, tpr, _ = roc_curve(y_test, res['prob'])
    axes[1].plot(fpr, tpr, label=f"{name} ({res['auc']:.3f})", linewidth=2)
axes[1].plot([0,1],[0,1],'k--')
axes[1].set_title('ROC Curves', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].set_xlabel('FPR'); axes[1].set_ylabel('TPR')

# Feature Importance
best_model = results[best_name]['model']
if hasattr(best_model, 'feature_importances_'):
    imp = pd.Series(best_model.feature_importances_, index=X_train.columns).sort_values(ascending=True)
    colors = plt.cm.Reds(np.linspace(0.3, 0.9, len(imp)))
    axes[2].barh(imp.index, imp.values, color=colors)
    axes[2].set_title('Feature Importances\n(Burnout Prediction)', fontweight='bold')

plt.tight_layout()
os.makedirs('outputs', exist_ok=True)
plt.savefig('outputs/10_burnout_model.png', bbox_inches='tight')
plt.show()

print("\n📋 Classification Report:")
print(classification_report(y_test, results[best_name]['pred'], target_names=['Low Risk','High Risk']))


In [ ]:
os.makedirs('models', exist_ok=True)
with open('models/burnout_model.pkl','wb') as f:
    pickle.dump(results[best_name]['model'], f)
print(f"✅ Saved models/burnout_model.pkl  ({best_name})")
